# D2 · M1.4 + M2.2 — Vector Stores, Part 1: Embeddings & Chroma

**Part 1 of 2.** This notebook covers M1.4 (embeddings & semantic search, graded) and the Chroma
half of M2.2. Qdrant and the store-comparison come in a follow-up notebook next session.

**Worked side by side with the Concept HTML page.** Read a short concept section, run the
matching cells below, go back to the page for the next concept — three round trips in this part.

**THE SITUATION:** Before a vector database means anything, you need to see the problem it
solves. This notebook builds an in-memory semantic search by hand first (M1.4), then stands up a
real vector store — Chroma — and shows the same search working at real scale, with metadata
filtering added in (M2.2, part 1).

**This notebook is fully working code — nothing to write.** Run each cell in order.

Concept page: `Day2/concept/D2_M1.4_M2.2_VectorStores_Concept.html`


## Setup

Loads your API key and the Meridian Retail Bank knowledge base (10 passages, 5 fixed test
queries). Run this once before anything else.


In [ ]:
import json
import math
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Find the .env file wherever it lives: this folder, any folder above it, or the
# Day1/labs/starter/ folder SETUP.md originally told you to create it in. One key
# file for the whole repo -- you never need to copy it between Day folders.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing -- never hard-code a key in this notebook."
)

client = OpenAI()
MODEL_EMBED = "text-embedding-3-small"

_DAY1_DATA = Path.cwd().resolve().parents[2] / "Day1" / "data"
PASSAGES_PATH = _DAY1_DATA / "D1_M1.4_banking_passages.json"
QUERIES_PATH = _DAY1_DATA / "D1_M1.4_fixed_queries.json"


def get_embedding(text):
    response = client.embeddings.create(model=MODEL_EMBED, input=text)
    return response.data[0].embedding


def cosine_similarity(vec_a, vec_b):
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = math.sqrt(sum(a * a for a in vec_a))
    magnitude_b = math.sqrt(sum(b * b for b in vec_b))
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)


def load_passages():
    with open(PASSAGES_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def load_fixed_queries():
    with open(QUERIES_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


print("Setup complete -- embedding client and Meridian knowledge base ready.")


## Concept A1, on the page -- then come back here for Tasks 1 and 2

Read **Concept A1 - Meaning Becomes Geometry** on the concept page, then run the cell below.


In [ ]:
WARMUP_SENTENCES = [
    "The cat sat quietly on the windowsill all afternoon.",
    "A dog ran happily around the park chasing a ball.",
    "Meridian Retail Bank increased its savings account interest rate this quarter.",
]
WARMUP_QUERY = "What is the current interest rate on savings accounts?"

print("=" * 72)
print("TASK 1 -- WARM-UP: ranking 3 sentences against 1 query")
print("=" * 72)
query_vec = get_embedding(WARMUP_QUERY)
warmup_ranking = []
for sentence in WARMUP_SENTENCES:
    vec = get_embedding(sentence)
    score = cosine_similarity(query_vec, vec)
    warmup_ranking.append({"sentence": sentence, "score": score})
warmup_ranking.sort(key=lambda r: r["score"], reverse=True)
for r in warmup_ranking:
    print("  {:.4f}  |  {}".format(r["score"], r["sentence"]))


def build_passage_index(passages):
    index = []
    for p in passages:
        vec = get_embedding(p["text"])
        index.append({"id": p["id"], "category": p["category"], "text": p["text"], "embedding": vec})
    return index

print("\nBuilding the in-memory passage index (10 embedding calls)...")
passages = load_passages()
fixed_queries = load_fixed_queries()
passage_index = build_passage_index(passages)
print("Indexed {} passages.".format(len(passage_index)))


**Notice:** the cat/dog sentences scored low, the savings-rate sentence scored high -- the
model never saw the word "interest" match anything, it matched *meaning*. Now go back to the
concept page for **Concept A2**.


## Concept A2, on the page -- then come back here for Tasks 4 and 5

Read **Concept A2 - Semantic vs. Keyword**, then run the cell below. **Task 5 is this module's
graded deliverable.**


In [ ]:
def semantic_search(query, index, top_k=3):
    query_vec = get_embedding(query)
    scored = [
        {"id": entry["id"], "text": entry["text"], "score": cosine_similarity(query_vec, entry["embedding"])}
        for entry in index
    ]
    scored.sort(key=lambda r: r["score"], reverse=True)
    return scored[:top_k]


def keyword_search(query, passages, top_k=3):
    query_words = set(query.lower().split())
    scored = []
    for p in passages:
        passage_words = set(p["text"].lower().split())
        overlap = len(query_words & passage_words)
        scored.append({"id": p["id"], "text": p["text"], "score": overlap})
    scored.sort(key=lambda r: r["score"], reverse=True)
    return scored[:top_k]


print("=" * 72)
print("TASK 4 -- SEMANTIC vs. KEYWORD search, same fixed queries")
print("=" * 72)
semantic_vs_keyword = []
for q in fixed_queries:
    sem_top = semantic_search(q["query"], passage_index, top_k=1)[0]
    kw_top = keyword_search(q["query"], passages, top_k=1)[0]
    sem_correct = sem_top["id"] == q["expected_top_id"]
    kw_correct = kw_top["id"] == q["expected_top_id"]
    semantic_vs_keyword.append({
        "query": q["query"], "expected_top_id": q["expected_top_id"],
        "semantic_top_id": sem_top["id"], "semantic_correct": sem_correct,
        "keyword_top_id": kw_top["id"], "keyword_correct": kw_correct,
    })
    print("\n\"{}\"".format(q["query"]))
    print("  semantic -> id {}  ({})".format(sem_top["id"], "correct" if sem_correct else "WRONG"))
    print("  keyword  -> id {}  ({})".format(kw_top["id"], "correct" if kw_correct else "WRONG"))

print("\n" + "=" * 72)
print("TASK 5 -- FIXED QUERY EVALUATION (graded)")
print("=" * 72)
evaluation_report = []
for q in fixed_queries:
    results = semantic_search(q["query"], passage_index, top_k=3)
    top_id = results[0]["id"]
    correct = top_id == q["expected_top_id"]
    evaluation_report.append({
        "query": q["query"], "expected_top_id": q["expected_top_id"],
        "retrieved_top_id": top_id, "top_3_ids": [r["id"] for r in results], "pass": correct,
    })
    marker = "PASS" if correct else "FAIL"
    print("\n\"{}\"".format(q["query"]))
    print("  top-3 ids: {}  (expected #1: {})  [{}]".format([r["id"] for r in results], q["expected_top_id"], marker))

passed_count = sum(1 for r in evaluation_report if r["pass"])
print("\n{}/{} fixed queries retrieved their expected passage at #1.".format(passed_count, len(evaluation_report)))


**Notice:** the pass/fail count above is M1.4's actual graded result. Now go back to the
concept page for **Concept B**.


## Concept B, on the page -- then come back here for the Chroma lab

Read **Concept B - From Python List to Real Vector Store**, then run the cell below. This
reuses the exact same embeddings from Task 2 -- nothing is re-computed, only re-indexed.


In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="meridian_banking_passages")

collection.add(
    ids=[str(p["id"]) for p in passage_index],
    embeddings=[p["embedding"] for p in passage_index],
    documents=[p["text"] for p in passage_index],
    metadatas=[{"category": p["category"]} for p in passage_index],
)
print("Ingested {} passages into Chroma, each tagged with a 'category' metadata field.".format(collection.count()))

query_text = "Someone used my account without my permission."
query_vec = get_embedding(query_text)
unfiltered = collection.query(query_embeddings=[query_vec], n_results=3)
print("\nQuery: \"{}\" -- no filter".format(query_text))
for pid, dist in zip(unfiltered["ids"][0], unfiltered["distances"][0]):
    print("  id {}  (distance {:.4f})".format(pid, dist))

filtered = collection.query(query_embeddings=[query_vec], n_results=3, where={"category": "fraud"})
print("\nSame query -- filtered to category='fraud'")
for pid, dist in zip(filtered["ids"][0], filtered["distances"][0]):
    print("  id {}  (distance {:.4f})".format(pid, dist))

chroma_query_results = {
    "query": query_text,
    "unfiltered_ids": unfiltered["ids"][0],
    "filtered_ids": filtered["ids"][0],
    "collection_count": collection.count(),
}


**Notice:** the filtered query only ever considers passages tagged `category: fraud` --
Chroma checked the metadata before it even compared vectors. Task 2-5's plain Python list had no
way to do that short of writing your own filter loop.

## What's next

Qdrant (a self-hosted, production-grade vector store) and a side-by-side comparison of Chroma
vs. Qdrant come in **Part 2**, next session -- this notebook's job was just to get a real vector
store up and running with metadata filtering. Run the final cell below to save your results.


In [ ]:
key_takeaway = """
KEY TAKEAWAY: Tasks 1-5 built semantic search by hand with a Python list and a
for-loop -- fine at 10 passages, not at scale. Chroma replaces that loop with a
real indexed store, and adds something the list never could: filtering by
metadata (category='fraud') before comparing vectors at all. Same embeddings,
better storage. Part 2 adds Qdrant and compares the two.
"""
print(key_takeaway)

results = {
    "warmup_ranking": warmup_ranking,
    "semantic_vs_keyword": semantic_vs_keyword,
    "evaluation_report": evaluation_report,
    "chroma_query_results": chroma_query_results,
    "key_takeaway": key_takeaway,
}

with open("m1_4_m2_2_part1_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m1_4_m2_2_part1_results.json --", len(json.dumps(results)), "bytes")
